<a href="https://colab.research.google.com/github/zgander/flyrank_ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zgander/flyrank_ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
# ============================================================
# W03 — DuckDB + Hugging Face Warehouse Setup
# ============================================================

!pip -q install duckdb

import duckdb
from google.colab import userdata

# ------------------------------------------------------------
# 1. Create DuckDB connection
# ------------------------------------------------------------

con = duckdb.connect()

# ------------------------------------------------------------
# 2. Get Hugging Face token from Colab Secrets
# ------------------------------------------------------------

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN was not found.\n\n"
        "Go to Colab → Secrets → Add a new secret:\n"
        "Name: HF_TOKEN\n"
        "Value: your Hugging Face READ token"
    )

# ------------------------------------------------------------
# 3. Store token securely in DuckDB
# ------------------------------------------------------------

con.execute(
    "SET VARIABLE hf_token = ?",
    [hf_token]
)

con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

# ------------------------------------------------------------
# 4. Define warehouse paths
# ------------------------------------------------------------

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"

DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

# Feature window: February 2026
FEB = f"{FACT}/month=2026-02/*.parquet"

# Label window: March 2026
MAR = f"{FACT}/month=2026-03/*.parquet"

print("✓ DuckDB connected")
print("✓ Hugging Face authentication configured")
print("✓ Warehouse paths configured")
print()
print("Feature window:", "February 2026")
print("Label window:", "March 2026")

✓ DuckDB connected
✓ Hugging Face authentication configured
✓ Warehouse paths configured

Feature window: February 2026
Label window: March 2026


## 1. Unit of analysis + time window

**Unit of analysis:** One row represents one content item for one client (`client_hash_id` × `content_hash_id`).

The source table, `fact_content_daily_performance`, is at daily grain: one row per client × content × day. For this project, I aggregate those daily observations into one row per client × content for a defined decision point.

**Feature window:** February 2026 (`2026-02-01` → `2026-02-28`). These are the signals that would have been knowable at the decision point.

**Label window:** March 2026 (`2026-03-01` → `2026-03-31`). This is the subsequent outcome window used to evaluate the page after the decision point.

The feature and label windows do not overlap. June 2026 is kept as a sealed final month rather than being used while developing the label logic.


In [4]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM read_parquet('{FACT}/month=2026-02/*.parquet')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate client × content × day keys:")
display(grain_check)

assert grain_check.empty, "Duplicate grain keys found — investigate before aggregating."

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate client × content × day keys:


,report_date,client_hash_id,content_hash_id,n


## 2. Fields: feature / label / context / excluded

### Feature

The features are signals that would be available at the decision point, using the February 2026 feature window:

- `gsc_impressions` — total Google Search Console impressions during February.
- `gsc_clicks` — total GSC clicks during February.
- `gsc_ctr` — aggregate click-through rate during February, calculated as clicks / impressions.
- `gsc_avg_position` — average search position during February.
- `content_age_days` — age of the content at the decision point.

These features describe the page's observed search visibility, traffic, and age before the label window.

### Label

The label is:

- `went_dark` — whether the content item received zero measured GSC clicks during March 2026, among content items for which GSC data was available during the label window.

March is strictly after the February feature window, so the label represents a subsequent observed outcome rather than an input available at prediction time.

### Context

The following fields are retained for grouping, joining, validation, and interpretation but are not model features:

- `client_hash_id` — identifies the pseudonymized client.
- `content_hash_id` — identifies the pseudonymized content item.
- `report_date` — identifies the daily observation and is used to construct the feature and label windows.

The identifiers are used for grouping and joins only, not as predictive features.

### Excluded

The following fields are excluded from the model:

- `trend_direction` — excluded because it is derived from `trend_pct` and therefore would duplicate information used to define a decline-related label.
- `trend_pct` — excluded because it is a derived trend signal and can introduce leakage depending on how its window is constructed.
- March GSC/GA4 metrics — excluded because they occur after the February decision point and therefore would expose future information.
- Product-derived scores such as `health_score` or `priority_score` — excluded because they represent pre-existing/product-generated decisions rather than independent observable signals.
- `action_type` — excluded because it represents an action/decision rather than an input available for predicting the outcome.

In [5]:
# Verify that the fields we plan to use actually exist
# and inspect their data types.

columns_check = con.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{FEB}')
""").df()

display(
    columns_check[
        columns_check["column_name"].isin([
            "client_hash_id",
            "content_hash_id",
            "report_date",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "ga4_sessions",
            "gsc_data_available",
            "ga4_data_available"
        ])
    ]
)

# Confirm that the key fields needed for the contract are present.

required_fields = {
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "gsc_data_available"
}

available_fields = set(columns_check["column_name"])

missing_fields = required_fields - available_fields

print("Required fields:", required_fields)
print("Missing fields:", missing_fields)

assert not missing_fields, f"Missing required fields: {missing_fields}"

print("✓ All required contract fields are present.")

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
10,gsc_avg_position,DOUBLE,YES,None,None,None
12,ga4_sessions,BIGINT,YES,None,None,None


Required fields: {'gsc_impressions', 'gsc_data_available', 'content_hash_id', 'gsc_clicks', 'gsc_avg_position', 'client_hash_id', 'report_date'}
Missing fields: set()
✓ All required contract fields are present.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# Section 3 — Verify the data contract with DuckDB queries

print("=" * 70)
print("1. GRAIN CHECK — client × content × day")
print("=" * 70)

grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM read_parquet('{FEB}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

display(grain_check)

assert grain_check.empty, "Duplicate client × content × day rows found."
print("✓ No duplicate client × content × day keys found.")


print("\n" + "=" * 70)
print("2. FEATURE WINDOW — February 2026")
print("=" * 70)

feb_window = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{FEB}')
""").df()

display(feb_window)


print("\n" + "=" * 70)
print("3. LABEL WINDOW — March 2026")
print("=" * 70)

mar_window = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{MAR}')
""").df()

display(mar_window)


print("\n" + "=" * 70)
print("4. GSC DATA AVAILABILITY — February")
print("=" * 70)

feb_availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS FALSE
        ) AS gsc_unavailable_rows
    FROM read_parquet('{FEB}')
""").df()

display(feb_availability)


print("\n" + "=" * 70)
print("5. GSC DATA AVAILABILITY — March")
print("=" * 70)

mar_availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS FALSE
        ) AS gsc_unavailable_rows
    FROM read_parquet('{MAR}')
""").df()

display(mar_availability)


print("\n" + "=" * 70)
print("6. MISSING VALUES — February feature fields")
print("=" * 70)

missing_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE gsc_impressions IS NULL
        ) AS missing_gsc_impressions,

        COUNT(*) FILTER (
            WHERE gsc_clicks IS NULL
        ) AS missing_gsc_clicks,

        COUNT(*) FILTER (
            WHERE gsc_avg_position IS NULL
        ) AS missing_gsc_avg_position,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS NULL
        ) AS missing_gsc_availability_flag

    FROM read_parquet('{FEB}')
""").df()

display(missing_check)


print("\n" + "=" * 70)
print("7. FEATURE/LABEL WINDOW OVERLAP CHECK")
print("=" * 70)

overlap_check = con.sql(f"""
    SELECT
        COUNT(*) AS overlapping_dates
    FROM (
        SELECT DISTINCT report_date
        FROM read_parquet('{FEB}')
    ) feb
    INNER JOIN (
        SELECT DISTINCT report_date
        FROM read_parquet('{MAR}')
    ) mar
    ON feb.report_date = mar.report_date
""").df()

display(overlap_check)

assert overlap_check.iloc[0]["overlapping_dates"] == 0, \
    "Feature and label windows overlap."

print("✓ February and March windows do not overlap.")


print("\n" + "=" * 70)
print("8. FINAL WINDOW SUMMARY")
print("=" * 70)

print("Feature window : 2026-02-01 → 2026-02-28")
print("Label window   : 2026-03-01 → 2026-03-31")
print("Feature grain  : client × content")
print("Raw fact grain : client × content × day")
print("✓ Section 3 verification complete.")

1. GRAIN CHECK — client × content × day


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n


✓ No duplicate client × content × day keys found.

2. FEATURE WINDOW — February 2026


,rows,clients,content_items,min_date,max_date
0,7355108,54,321546,2026-02-01,2026-02-28


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,missing_gsc_impressions,missing_gsc_clicks,missing_gsc_avg_position,missing_gsc_availability_flag
0,7355108,92256,92256,4733326,92256



7. FEATURE/LABEL WINDOW OVERLAP CHECK


,overlapping_dates
0,0


✓ February and March windows do not overlap.

8. FINAL WINDOW SUMMARY
Feature window : 2026-02-01 → 2026-02-28
Label window   : 2026-03-01 → 2026-03-31
Feature grain  : client × content
Raw fact grain : client × content × day
✓ Section 3 verification complete.


## 4. Data limits

### Data limits

This dataset cannot tell us whether refreshing a page will *cause* its performance to improve. Our label is an observed future outcome, not a causal measurement of the effect of a refresh.

The warehouse is an **unbalanced panel**: different clients have different amounts of historical data. Therefore, not every client or content item necessarily has sufficient observations in both the feature and label windows. The resulting dataset may therefore represent only the subset with adequate measured history.

Some rows contain **GSC data without GA4 data** because GA4 tracking starts at different times for different clients. A GA4 value of zero before `ga4_data_start` must not be interpreted as zero engagement. The `ga4_data_available` flag must be used when GA4 features are included.

The February feature window and March label window must remain strictly separated. Any feature derived from March or later would expose future information and create data leakage.

The data also cannot establish why a page's performance changed. We observe search and engagement outcomes, but we do not observe every external factor that may have influenced them.

Finally, the model can identify pages that resemble pages with a particular historical outcome, but it cannot guarantee that the recommended action (refresh, expand, protect, prune, or monitor) will produce that outcome.

In [7]:
# Section 4 — Verify important data limitations

print("=" * 70)
print("1. CLIENT HISTORY COVERAGE")
print("=" * 70)

client_history = con.sql(f"""
    SELECT
        COUNT(*) AS clients,
        COUNT(*) FILTER (
            WHERE gsc_data_start <= DATE '2026-02-01'
        ) AS clients_with_gsc_before_feature_window,
        COUNT(*) FILTER (
            WHERE gsc_data_start > DATE '2026-02-01'
        ) AS clients_starting_after_feature_window
    FROM read_parquet('{DIM_CLIENTS}')
""").df()

display(client_history)


print("=" * 70)
print("2. GA4 AVAILABILITY IN FEBRUARY")
print("=" * 70)

ga4_availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS FALSE
        ) AS ga4_unavailable_rows
    FROM read_parquet('{FEB}')
""").df()

display(ga4_availability)


print("=" * 70)
print("3. FEATURE / LABEL WINDOW SEPARATION")
print("=" * 70)

windows = con.sql(f"""
    SELECT
        MIN(report_date) AS feature_start,
        MAX(report_date) AS feature_end
    FROM read_parquet('{FEB}')
""").df()

label_windows = con.sql(f"""
    SELECT
        MIN(report_date) AS label_start,
        MAX(report_date) AS label_end
    FROM read_parquet('{MAR}')
""").df()

display(windows)
display(label_windows)

feature_end = windows.iloc[0]["feature_end"]
label_start = label_windows.iloc[0]["label_start"]

assert feature_end < label_start, (
    "Feature and label windows overlap — potential leakage."
)

print("✓ Feature window ends before label window begins.")


1. CLIENT HISTORY COVERAGE


,clients,clients_with_gsc_before_feature_window,clients_starting_after_feature_window
0,104,41,26


2. GA4 AVAILABILITY IN FEBRUARY


,total_rows,ga4_available_rows,ga4_unavailable_rows
0,7355108,145321,3057464


3. FEATURE / LABEL WINDOW SEPARATION


,feature_start,feature_end
0,2026-02-01,2026-02-28


,label_start,label_end
0,2026-03-01,2026-03-31


✓ Feature window ends before label window begins.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.